# Subtitle Overlay STT - Colab Inference Server

Este notebook corre Faster Whisper en GPU de Colab y expone un servidor HTTP para recibir audio desde la PC local.

## Setup:
1. Runtime → Change runtime type → **T4 GPU**
2. Correr todas las celdas en orden
3. Copiar la URL pública que aparece al final
4. En tu PC local, correr: `STT_COLAB_URL=<url> ./scripts/run_stt_colab.sh`

In [ ]:
# Install dependencies
!pip install -q faster-whisper flask pyngrok

## ⚠️ Configurar ngrok authtoken

ngrok requiere autenticación (gratis). Opciones:

**Opción A: ngrok (recomendado)**
1. Crear cuenta: https://dashboard.ngrok.com/signup
2. Obtener token: https://dashboard.ngrok.com/get-started/your-authtoken
3. Descomentar y ejecutar la celda de abajo:

**Opción B: Colab Tunnel (no requiere cuenta)**
- Usar la celda alternativa al final que usa `google.colab` built-in tunneling

In [ ]:
# DESCOMENTAR Y PONER TU TOKEN:
# from pyngrok import ngrok
# ngrok.set_auth_token("TU_NGROK_TOKEN_AQUI")

In [ ]:
# Verify GPU is available
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from faster_whisper import WhisperModel
import time

# Configuration - same defaults as stt_receiver.py
MODEL_SIZE = "small"  # Change to 'base' or 'medium' if you want
DEVICE = "cuda"
COMPUTE_TYPE = "float16"  # GPU uses float16, not int8
LANGUAGE = "es"
BEAM_SIZE = 5
VAD_FILTER = True

print(f"Loading Faster Whisper model={MODEL_SIZE} device={DEVICE} compute_type={COMPUTE_TYPE}")
t0 = time.time()
model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)
print(f"✅ Model loaded in {time.time() - t0:.1f}s")

In [ ]:
# Hallucination detection - same as stt_receiver.py
HALLUCINATION_MARKERS = (
    "amara.org",
    "subtítulos realizados por",
    "subtitulos realizados por",
    "suscríbete al canal",
    "gracias por ver el video",
    "gracias por ver este video",
)

def is_hallucination(text):
    lowered = text.lower()
    return any(marker in lowered for marker in HALLUCINATION_MARKERS)

def transcribe_audio(audio_float32, start_sec, end_sec, is_final, seq):
    """Transcribe audio chunk and return event dict."""
    t0 = time.time()
    segments, _info = model.transcribe(
        audio_float32,
        language=LANGUAGE,
        beam_size=BEAM_SIZE,
        vad_filter=VAD_FILTER,
    )
    text = " ".join(segment.text.strip() for segment in segments).strip()
    elapsed = time.time() - t0
    
    if not text:
        return None, elapsed
    
    if is_hallucination(text):
        print(f"Dropping hallucination: {text!r}")
        return None, elapsed
    
    event = {
        "seq": seq,
        "is_final": bool(is_final),
        "start_sec": round(start_sec, 3),
        "end_sec": round(end_sec, 3),
        "text": text,
    }
    return event, elapsed

print("✅ Transcription engine ready")

In [ ]:
from flask import Flask, request, jsonify
import numpy as np
import base64

app = Flask(__name__)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({"status": "ok", "model": MODEL_SIZE, "device": DEVICE})

@app.route('/transcribe', methods=['POST'])
def transcribe():
    """
    Expects JSON:
    {
        "audio_base64": "<base64-encoded float32 array>",
        "start_sec": 0.0,
        "end_sec": 1.5,
        "is_final": true,
        "seq": 0
    }
    
    Returns:
    {
        "event": {"seq": 0, "is_final": true, "start_sec": 0.0, "end_sec": 1.5, "text": "hola mundo"},
        "inference_sec": 0.123
    }
    or {"event": null, "inference_sec": 0.05} if empty/hallucination
    """
    try:
        data = request.get_json()
        audio_bytes = base64.b64decode(data['audio_base64'])
        audio_float32 = np.frombuffer(audio_bytes, dtype=np.float32)
        start_sec = data['start_sec']
        end_sec = data['end_sec']
        is_final = data['is_final']
        seq = data['seq']
        
        event, inference_sec = transcribe_audio(audio_float32, start_sec, end_sec, is_final, seq)
        
        return jsonify({
            "event": event,
            "inference_sec": round(inference_sec, 3)
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 500

print("✅ Flask app ready")

## 🚀 Start Server (Opción A: ngrok)

**Requiere:** ngrok authtoken configurado arriba

In [ ]:
# Start ngrok tunnel and Flask server
from pyngrok import ngrok
import threading

# Start Flask in background thread
def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

# Wait a bit for Flask to start
time.sleep(2)

# Start ngrok tunnel
tunnel = ngrok.connect(5000)
public_url = str(tunnel.public_url)  # Extract clean URL string
print("\n" + "="*80)
print("🚀 Colab Inference Server Running!")
print("="*80)
print(f"\nPublic URL: {public_url}")
print(f"\nEn tu PC, corre:")
print(f"  STT_COLAB_URL={public_url} ./scripts/run_stt_colab.sh")
print("\nPresiona Ctrl+C para detener (pero el notebook seguirá activo)")
print("="*80 + "\n")

# Keep the cell running
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nStopping server...")
    ngrok.disconnect(tunnel.public_url)

## 🚀 Start Server (Opción B: localtunnel)

**Alternativa sin ngrok authtoken** - usa localtunnel (puede ser menos estable)

In [ ]:
# Alternative: localtunnel (no auth required)
!npm install -g localtunnel

import threading
import subprocess
import time

# Start Flask
def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(2)

# Start localtunnel
print("\n" + "="*80)
print("🚀 Colab Inference Server Running (localtunnel)")
print("="*80)
print("\nEsperando URL pública...\n")
!lt --port 5000